# **Data loading**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#change to your path to the shared folder
data_path ="/content/drive/MyDrive/NLP Team Project - SkillShift/job posting data"

In [ ]:
import os

os.listdir(data_path)

In [ ]:
import pandas as pd

# ===== 1. posting(main columns) =====
postings = pd.read_csv(f"{data_path}/merged_postings.csv")

# ===== 2. skills  =====
job_skills = pd.read_csv(f"{data_path}/merged_job_skills.csv")
skills = pd.read_csv(f"{data_path}/merged_skills.csv")

# ===== 3. industry  =====
job_industries = pd.read_csv(f"{data_path}/merged_job_industries.csv")
industries = pd.read_csv(f"{data_path}/merged_industries.csv")


# ===== 4. companies =====
companies = pd.read_csv(f"{data_path}/merged_companies.csv")

# ===== print and check =====
print("postings:", postings.shape)
print("job_skills:", job_skills.shape)
print("skills:", skills.shape)
print("job_industries:", job_industries.shape)
print("industries:", industries.shape)
print("companies:", companies.shape)

In [ ]:
postings.head()

# **Data Merge**
Goal: Merge job postings with skills and industry data to create a unified dataset for analysis. First of all, we need to figure out the structure and the information density of each columns, so that we can decide to keep which columns.

In [ ]:
# All columns of postings.
postings.columns

In [ ]:
# Percentage of missing values of each column
postings.isna().mean().sort_values(ascending=False)

In [ ]:
# Count number of different experience levels.
postings["formatted_experience_level"].value_counts()

In [ ]:
#keep only relevant columns of postings file
cols_to_keep = [
    "job_id",
    "title",
    "description",
    "company_name",
    "location",
    "formatted_experience_level",
    "original_listed_time",
    "original_listed_month",
    "original_listed_year",
    "remote_allowed",
    "work_type",
    "formatted_work_type"
]

postings = postings[[col for col in cols_to_keep if col in postings.columns]]
postings = postings[[col for col in cols_to_keep if col in postings.columns]]

print(postings.shape)

Merge job_skills with skills to map skill abbreviations to readable skill names. Then group by job_id to create a list of skills for each job

In [ ]:
job_skills_merged=job_skills.merge(skills, on='skill_abr', how='left')
job_skills_merged.head()

In [ ]:
job_skills_grouped = job_skills_merged.groupby("job_id")\
 ["skill_name"].apply(list).reset_index()
job_skills_grouped.head()

In [ ]:
# Merge job skills info to the posting dataframe
postings_with_skills = postings.merge(
    job_skills_grouped,
    on="job_id",
    how="left"
)

In [ ]:
# Merge industry info to posting, and get the final dataframe.
job_industries_merged = job_industries.merge(
    industries,
    on="industry_id",
    how="left"
)
job_industries_grouped = job_industries_merged.groupby("job_id")\
 ["industry_name"].apply(list).reset_index()
final_df = postings_with_skills.merge(
    job_industries_grouped,
    on="job_id",
    how="left"
)
print(final_df.columns)
print(final_df.shape)

In [ ]:
subset_cols = ['title', 'description']

final_df = final_df.drop_duplicates(subset=subset_cols, keep='first')

final_df = final_df.reset_index(drop=True)

print(final_df.shape[0])

In [ ]:
# Check missing value ratio for each column
missing_ratio = final_df.isna().mean().sort_values(ascending=False)
missing_ratio

In [ ]:
# Only drop missing values for critical fields (e.g., description),
# while retaining other observations to preserve data coverage
final_df = final_df.dropna(subset=["description"])
final_df.shape

# Summary Stats (EDA)

Generate summary statistics to understand data distribution and support AI exposure definition

In [ ]:
from collections import Counter

all_skills = final_df["skill_name"].dropna().explode()
Counter(all_skills).most_common(20)


In [ ]:
all_industries = final_df["industry_name"].dropna().explode()
Counter(all_industries).most_common(20)

In [ ]:
print("Total jobs:", final_df.shape[0])
print("Jobs with description:", final_df["description"].notna().sum())
print("Jobs with skills:", final_df["skill_name"].notna().sum())
print("Jobs with industries:", final_df["industry_name"].notna().sum())


In [ ]:
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Flatten the list of skills and count the top 10
# Use .dropna() to handle potential NaN entries before exploding
all_skills = final_df['skill_name'].dropna().explode()
top_skills = Counter(all_skills).most_common(10)
skills_df = pd.DataFrame(top_skills, columns=['Skill', 'Count'])

plt.figure(figsize=(8,5))
ax = sns.barplot(
    x='Count',
    y='Skill',
    data=skills_df,
    palette="viridis"
)

# 添加数值标签
for i, v in enumerate(skills_df['Count']):
    ax.text(v + 50, i, str(v), va='center')

plt.title("Top 10 Most Common Skills in Job Postings", fontsize=14)
plt.xlabel("Frequency")
plt.ylabel("Skill")

plt.tight_layout()
plt.show()

In [ ]:
final_df['formatted_experience_level'].value_counts(normalize=True)

In [ ]:
exp_dist = final_df["formatted_experience_level"].value_counts(normalize=True)

plt.figure(figsize=(8,5))

ax = sns.barplot(
    x=exp_dist.index,
    y=exp_dist.values,
    palette="Blues_r"
)

# 添加百分比标签（在柱子上方）
for i, v in enumerate(exp_dist.values):
    ax.text(i, v + 0.01, f"{v:.1%}", ha='center')

plt.title("Distribution of Job Postings by Experience Level", fontsize=14)
plt.xlabel("Experience Level")
plt.ylabel("Percentage")

plt.ylim(0, max(exp_dist.values) * 1.2)

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 industries
industry_counts = final_df['industry_name'].explode().value_counts().head(10)

plt.figure(figsize=(10,5))
ax = sns.barplot(
    x=industry_counts.values,
    y=industry_counts.index,
    palette="magma"
)

# 添加数值标签
for i, v in enumerate(industry_counts.values):
    ax.text(v + 100, i, str(v), va='center')

plt.title("Top 10 Industries by Job Postings", fontsize=14)
plt.xlabel("Number of Jobs")
plt.ylabel("Industry")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

industry_counts = (
    final_df['industry_name']
    .explode()
    .value_counts()
)

top5 = industry_counts.head(5)
others = industry_counts.iloc[5:].sum()

labels = list(top5.index) + ['Others']
sizes = list(top5.values) + [others]

colors = [
    "#4C78A8",
    "#F58518",
    "#54A24B",
    "#E45756",
    "#B279A2",
    "#CCCCCC"
]
plt.figure(figsize=(6,6))
plt.pie(
    sizes,
    autopct='%1.1f%%',
    startangle=140,
    labels=labels,
    colors=colors,
    textprops={'fontsize': 12}
)

plt.title("Distribution of job postings across industries (Top 5 + Others)")
plt.tight_layout()
plt.show()

In [ ]:
import re
from collections import Counter

all_text = " ".join(final_df["description"].dropna()).lower()

words = re.findall(r'\b[a-z]{2,}\b', all_text)

from wordcloud import STOPWORDS

stopwords = set(STOPWORDS)

custom_stopwords = {
    "job", "jobs", "work", "working", "role", "position",
    "company", "team", "experience", "required", "looking"
}

stopwords = stopwords.union(custom_stopwords)

filtered_words = [w for w in words if w not in stopwords]

In [ ]:
word_counts = Counter(filtered_words)

counts = sorted(word_counts.values(), reverse=True)
import numpy as np

ranks = np.arange(1, len(counts) + 1)
import matplotlib.pyplot as plt

plt.figure(figsize=(6,5))

plt.loglog(ranks, counts, marker='o')

plt.title("Zipf's Law: Word Frequency vs Rank")
plt.xlabel("Rank (log scale)")
plt.ylabel("Frequency (log scale)")

plt.grid(True, which="both", linestyle='--', linewidth=0.5)

plt.show()

#Annotation of AI Exposure Levels for Job Roles


## AI Exposure Measurement using TF-IDF + Similarity

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ai_keywords_df = pd.read_csv("/content/drive/MyDrive/NLP Team Project - SkillShift/ai_skill_list.csv")
ai_ref_text = " ".join(ai_keywords_df['ai_skill'].astype(str))

# Combine all job descriptions with the AI reference text
# The last entry in this list will represent the "AI concept vector"
all_texts = final_df['description'].tolist() + [ai_ref_text]

# Apply TF-IDF vectorization
tfidf_vect = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vect.fit_transform(all_texts)

ai_vec = tfidf_matrix[-1:]

jd_vecs = tfidf_matrix[:-1]

exposure_scores = cosine_similarity(jd_vecs, ai_vec)

final_df['ai_exposure_tfidf'] = exposure_scores.flatten()

display(final_df[['title', 'ai_exposure_tfidf']].sort_values(by='ai_exposure_tfidf', ascending=False).head(10))

## Convert AI Exposure Score into Categories

In [ ]:
import numpy as np

positive_scores = final_df[final_df['ai_exposure_tfidf'] > 0]['ai_exposure_tfidf']

threshold = positive_scores.quantile(0.75)

def label_exposure(score):
    if score == 0:
        return 'non-ai expose'
    elif score >= threshold:
        return 'high ai expose'
    else:
        return 'low ai expose'

final_df['ai_exposure_label'] = final_df['ai_exposure_tfidf'].apply(label_exposure)

print(final_df['ai_exposure_label'].value_counts())
print(f'Threshold is {threshold}')

display(final_df[['title', 'ai_exposure_tfidf', 'ai_exposure_label']].sort_values(by='ai_exposure_tfidf', ascending=False).head(10))

# Classification Model: Predicting AI Exposure

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

X = jd_vecs
y = final_df['ai_exposure_label'].apply(lambda x: 0 if x == 'non-ai expose' else 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Initial Model Performance:")
print(f"Accuracy: {acc:.4f}")
print(f"F1-score: {f1:.4f}")

if f1 < 0.8:
    print("\nF1-score < 0.8, no good")
    param_grid = {
        'C': [0.1, 1, 10, 100],
        'solver': ['liblinear', 'lbfgs']
    }
    grid_search = GridSearchCV(LogisticRegression(max_iter=1000, class_weight='balanced'),
                               param_grid, cv=5, scoring='f1')
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    y_pred_best = best_model.predict(X_test)

    print(f"Best Params: {grid_search.best_params_}")
    print(f"Optimized F1-score: {f1_score(y_test, y_pred_best):.4f}")
    print("\nFinal Classification Report:")
    print(classification_report(y_test, y_pred_best))
    model_to_use = best_model
else:
    print("\nF1-score is good")
    model_to_use = lr


# Data Visualization and Time Trend

## Monthly Job Posting Trends by AI Exposure Level

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

plot_df = final_df.copy()

plot_df['listing_month'] = pd.to_datetime(
    plot_df['original_listed_time'], errors='coerce', format='mixed'
).dt.to_period('M').astype(str)

plot_df = plot_df[plot_df['listing_month'] != 'NaT']


plot_df = plot_df[
    (plot_df['original_listed_year'] >= 2023) &
    (plot_df['original_listed_year'] <= 2024)
]

monthly_counts = (
    plot_df.groupby(['listing_month', 'ai_exposure_label'])
    .size()
    .reset_index(name='job_count')
)


sns.set_theme(style="whitegrid")
plt.figure(figsize=(14, 7))
line_plot = sns.lineplot(
    data=monthly_counts,
    x='listing_month',
    y='job_count',
    hue='ai_exposure_label',
    marker='o',
    linewidth=2.5,
    palette={
        'high ai expose': '#d62728',
        'low ai expose': '#1f77b4',
        'non-ai expose': '#2ca02c'
    }
)

plt.title('Monthly Trend: Volume of Job Postings by AI Exposure Type (2023-2024)', fontsize=16, pad=20)
plt.ylabel('Number of Job Postings', fontsize=12)
plt.xlabel('Listing Month', fontsize=12)


plt.legend(title='AI Exposure Level', title_fontsize='11', fontsize='10')

plt.show()



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast

# 1. Load the dataset (Assuming final_df is already defined in your environment)
df = final_df

# 2. Pre-processing: Handle industry lists and explode
def safe_parse(x):
    if isinstance(x, str) and x.startswith('['):
        try: return ast.literal_eval(x)
        except: return x
    return x

df['industry_name'] = df['industry_name'].apply(safe_parse)
df = df.explode('industry_name')

# 3. Pivot the data
pivot_df = (
    df.groupby(['industry_name', 'original_listed_year'])['ai_exposure_tfidf']
    .mean()
    .unstack()
)

# 4. Filter for industries present in BOTH 2023 and 2024
pivot_df = pivot_df.dropna(subset=[2023, 2024])

# 5. Calculate INCREASE (2024 value minus 2023 value)
pivot_df['increase'] = pivot_df[2024] - pivot_df[2023]
top_10_increases = pivot_df.sort_values(by='increase', ascending=False).head(10)

# 6. Prepare data for plotting
plot_data = top_10_increases.reset_index().melt(
    id_vars='industry_name',
    value_vars=[2023, 2024],
    var_name='Year',
    value_name='Avg AI Exposure'
)

# 7. Create the visualization
plt.figure(figsize=(12, 8))
sns.barplot(
    data=plot_data,
    x='Avg AI Exposure',
    y='industry_name',
    hue='Year',
    palette='colorblind'
)

# --- ADDED: Dotted line at the specific TF-IDF threshold ---
threshold_val = 0.028012214586667014
plt.axvline(x=threshold_val, color='black', linestyle=':', linewidth=2, label= "High AI Exposure Threshold")

plt.title('Top 10 Industries with Greatest INCREASE in AI Exposure (2023 to 2024)', fontsize=15)
plt.xlabel('Average AI Exposure (TF-IDF)')
plt.ylabel('Industry')
plt.grid(axis='x', linestyle='--', alpha=0.6)

# Update legend to include the threshold line label
plt.legend(title='Year', loc='lower right')

plt.tight_layout()
plt.savefig('top_10_increases_with_threshold.png')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast

# 1. Load the dataset
df = final_df

# 2. Handle lists and explode industry names
def safe_parse(x):
    if isinstance(x, str) and x.startswith('['):
        try: return ast.literal_eval(x)
        except: return x
    return x

df['industry_name'] = df['industry_name'].apply(safe_parse)
df = df.explode('industry_name')

# 3. Pivot to get years as columns and drop missing data
pivot_df = (
    df.groupby(['industry_name', 'original_listed_year'])['ai_exposure_tfidf']
    .mean()
    .unstack()
)

# Ensure we only compare industries present in both years
pivot_df = pivot_df.dropna(subset=[2023, 2024])

# 4. Calculate Increase (2024 - 2023) and get Top 10
pivot_df['increase'] = pivot_df[2024] - pivot_df[2023]
top_10_df = pivot_df.sort_values(by='increase', ascending=False).head(10)

# 5. Reshape for plotting (Melt)
plot_data = top_10_df.reset_index().melt(
    id_vars='industry_name',
    value_vars=[2023, 2024],
    var_name='Year',
    value_name='Avg AI Exposure'
)

# 6. Create the Line Plot
plt.figure(figsize=(10, 7))
sns.lineplot(
    data=plot_data,
    x='Year',
    y='Avg AI Exposure',
    hue='industry_name',
    marker='o',
    linewidth=3
)

# 7. Formatting
threshold_val = 0.028012214586667014
# Add a horizontal dotted line at the threshold
plt.axhline(y=threshold_val, color='black', linestyle=':', linewidth=2, label="High AI Exposure Threshold")

plt.title('Top 10 Industries: Steepest Increase in AI Exposure (2023-2024)', fontsize=15)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Average AI Exposure (TF-IDF)', fontsize=12)
plt.xticks([2023, 2024]) # Force clear year labels

# Update the legend to include the threshold line
plt.legend(title='Industry & Reference', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('top_10_increase_line_graph_with_threshold.png')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import pandas as pd
import numpy as np

plot_df = final_df.copy()
plot_df['is_high_ai'] = (plot_df['ai_exposure_label'] == 'high ai expose').astype(int)

plot_df = plot_df[
    plot_df['original_listed_year'].isin([2023, 2024]) &
    plot_df['formatted_experience_level'].notna()
]

level_stats = (
    plot_df.groupby('formatted_experience_level')
    .agg(total=('is_high_ai', 'count'), high_ai=('is_high_ai', 'sum'))
    .reset_index()
)
level_stats = level_stats[level_stats['total'] >= 30]
level_stats['high_ai_ratio'] = level_stats['high_ai'] / level_stats['total']

level_stats = level_stats.sort_values('high_ai_ratio', ascending=True)

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(9, 6))

colors = level_stats['high_ai_ratio'].apply(
    lambda x: '#d62728' if x >= level_stats['high_ai_ratio'].quantile(0.67)
              else ('#ff7f0e' if x >= level_stats['high_ai_ratio'].quantile(0.33)
              else '#1f77b4')
)

bars = ax.barh(
    level_stats['formatted_experience_level'],
    level_stats['high_ai_ratio'],
    color=colors,
    edgecolor='white',
    linewidth=0.8,
    height=0.6
)

for bar, (_, row) in zip(bars, level_stats.iterrows()):
    ax.text(
        bar.get_width() + 0.003,
        bar.get_y() + bar.get_height() / 2,
        f"{row['high_ai_ratio']:.1%}  (n={row['total']:,})",
        va='center', ha='left', fontsize=10
    )

ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_xlim(0, level_stats['high_ai_ratio'].max() + 0.12)
ax.set_title('% High-AI Exposure by Experience Level (2023–2024)',
             fontsize=13, pad=14)
ax.set_xlabel('% of Job Postings with High AI Exposure', fontsize=11)
ax.set_ylabel('Experience Level', fontsize=11)

avg = plot_df['is_high_ai'].mean()
ax.axvline(avg, color='gray', linestyle='--', linewidth=1.2, alpha=0.7)
ax.text(avg + 0.002, -0.6, f'Avg: {avg:.1%}',
        color='gray', fontsize=9, va='top')

plt.tight_layout()
plt.show()

print("\nDetail Data：")
print(level_stats[['formatted_experience_level', 'total', 'high_ai', 'high_ai_ratio']]
      .sort_values('high_ai_ratio', ascending=False)
      .to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import pandas as pd
import numpy as np

plot_df = final_df.copy()
plot_df['is_high_ai'] = (plot_df['ai_exposure_label'] == 'high ai expose').astype(int)

plot_df = plot_df[
    plot_df['original_listed_year'].isin([2023, 2024]) &
    plot_df['industry_name'].notna()
]

plot_df = plot_df.explode('industry_name')
plot_df = plot_df[plot_df['industry_name'].notna()]

industry_stats = (
    plot_df.groupby('industry_name')
    .agg(total=('is_high_ai', 'count'), high_ai=('is_high_ai', 'sum'))
    .reset_index()
)

industry_stats = industry_stats[industry_stats['total'] >= 100]
industry_stats['high_ai_ratio'] = industry_stats['high_ai'] / industry_stats['total']

top20 = industry_stats.sort_values('high_ai_ratio', ascending=False).head(20)
top20 = top20.sort_values('high_ai_ratio', ascending=True)

print(f"Total Number of Industries（Samples>=100）: {len(industry_stats)}")
print("\nTop 10 Industries of High AI Exposure")
print(top20.tail(10)[['industry_name', 'total', 'high_ai_ratio']].to_string(index=False))

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 10))

n = len(top20)
colors = ['#1f77b4'] * n
colors[-5:] = ['#ff7f0e'] * 5
colors[-3:] = ['#d62728'] * 3

bars = ax.barh(
    top20['industry_name'],
    top20['high_ai_ratio'],
    color=colors,
    edgecolor='white',
    linewidth=0.8,
    height=0.65
)

for bar, (_, row) in zip(bars, top20.iterrows()):
    ax.text(
        bar.get_width() + 0.003,
        bar.get_y() + bar.get_height() / 2,
        f"{row['high_ai_ratio']:.1%}  (n={row['total']:,})",
        va='center', ha='left', fontsize=9
    )

avg = plot_df['is_high_ai'].mean()
ax.axvline(avg, color='gray', linestyle='--', linewidth=1.2, alpha=0.7)
ax.text(avg + 0.002, 0.3, f'Avg: {avg:.1%}',
        color='gray', fontsize=9, va='bottom')

ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_xlim(0, top20['high_ai_ratio'].max() + 0.15)
ax.set_title('Top 20 Industries by % High-AI Exposure (2023–2024)',
             fontsize=13, pad=14)
ax.set_xlabel('% of Job Postings with High AI Exposure', fontsize=11)
ax.set_ylabel('Industry', fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import pandas as pd
import numpy as np

plot_df = final_df.copy()
plot_df['is_high_ai'] = (plot_df['ai_exposure_label'] == 'high ai expose').astype(int)

plot_df = plot_df[
    plot_df['original_listed_year'].isin([2023, 2024]) &
    plot_df['industry_name'].notna() &
    plot_df['formatted_experience_level'].notna()
]

# explode industry list
plot_df = plot_df.explode('industry_name')
plot_df = plot_df[plot_df['industry_name'].notna()]

industry_overall = (
    plot_df.groupby('industry_name')
    .agg(total=('is_high_ai', 'count'), high_ai=('is_high_ai', 'sum'))
    .reset_index()
)
industry_overall = industry_overall[industry_overall['total'] >= 100]
industry_overall['ratio'] = industry_overall['high_ai'] / industry_overall['total']
top_industries = industry_overall.nlargest(15, 'ratio')['industry_name'].tolist()
plot_df = plot_df[plot_df['industry_name'].isin(top_industries)]

heatmap_data = (
    plot_df.groupby(['industry_name', 'formatted_experience_level'])
    .agg(total=('is_high_ai', 'count'), high_ai=('is_high_ai', 'sum'))
    .reset_index()
)
heatmap_data = heatmap_data[heatmap_data['total'] >= 20]
heatmap_data['high_ai_ratio'] = heatmap_data['high_ai'] / heatmap_data['total']

level_order = ['Internship', 'Entry level', 'Associate',
               'Mid-Senior level', 'Director', 'Executive']
pivot = heatmap_data.pivot(
    index='industry_name',
    columns='formatted_experience_level',
    values='high_ai_ratio'
)

existing_levels = [l for l in level_order if l in pivot.columns]
pivot = pivot[existing_levels]

pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

print(f"Heatmap dimensions: {pivot.shape}  ({pivot.shape[0]} industrt x {pivot.shape[1]} experience level)")
print(pivot.round(3))

sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(11, 8))

annot_data = pivot.applymap(lambda x: f"{x:.1%}" if pd.notna(x) else "")

sns.heatmap(
    pivot,
    ax=ax,
    cmap='YlOrRd',
    annot=annot_data,
    fmt='',
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'format': mtick.PercentFormatter(xmax=1, decimals=0),
              'shrink': 0.8, 'label': '% High-AI Exposure'},
    vmin=0,
    vmax=pivot.max().max()
)

ax.set_title('% High-AI Exposure: Top 15 Industries × Experience Level (2023–2024)',
             fontsize=13, pad=16)
ax.set_xlabel('Experience Level', fontsize=11)
ax.set_ylabel('Industry', fontsize=11)
ax.tick_params(axis='x', rotation=30)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import pandas as pd
import numpy as np

plot_df = final_df.copy()
plot_df['is_high_ai'] = (plot_df['ai_exposure_label'] == 'high ai expose').astype(int)

plot_df = plot_df[
    plot_df['original_listed_year'].isin([2023, 2024]) &
    plot_df['industry_name'].notna()
]

plot_df = plot_df.explode('industry_name')
plot_df = plot_df[plot_df['industry_name'].notna()]


industry_overall = (
    plot_df.groupby('industry_name')
    .agg(total=('is_high_ai', 'count'), high_ai=('is_high_ai', 'sum'))
    .reset_index()
)
industry_overall = industry_overall[industry_overall['total'] >= 100]
industry_overall['ratio'] = industry_overall['high_ai'] / industry_overall['total']
top_industries = industry_overall.nlargest(10, 'ratio')['industry_name'].tolist()

plot_df = plot_df[plot_df['industry_name'].isin(top_industries)]

grouped = (
    plot_df.groupby(['original_listed_year', 'industry_name'])
    .agg(total=('is_high_ai', 'count'), high_ai=('is_high_ai', 'sum'))
    .reset_index()
)
grouped = grouped[grouped['total'] >= 30]
grouped['high_ai_ratio'] = grouped['high_ai'] / grouped['total']

grouped['year'] = grouped['original_listed_year'].astype(int)


sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 7))

colors = plt.cm.tab10.colors
industry_color = {ind: colors[i] for i, ind in enumerate(top_industries)}

for industry in top_industries:
    data = grouped[grouped['industry_name'] == industry].sort_values('year')
    if len(data) < 2:
        continue

    ax.plot(
        data['year'],
        data['high_ai_ratio'],
        marker='o',
        linewidth=2.5,
        markersize=9,
        label=industry,
        color=industry_color[industry]
    )

    for _, row in data.iterrows():
        offset = 6 if row['year'] == 2024 else -6
        ha = 'left' if row['year'] == 2024 else 'right'
        ax.annotate(
            f"{row['high_ai_ratio']:.1%}",
            xy=(row['year'], row['high_ai_ratio']),
            xytext=(offset, 4),
            textcoords='offset points',
            fontsize=8.5,
            ha=ha,
            color=industry_color[industry]
        )

ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=1))

ax.set_xticks([2023, 2024])
ax.set_xticklabels(['2023', '2024'])
ax.set_xlim(2022.5, 2024.5)

ax.set_title('% High-AI Exposure by Industry: 2023 vs 2024\n(Top 10 Industries)',
             fontsize=13, pad=14)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('% High-AI Job Postings', fontsize=11)
plt.legend(title='Industry', fontsize=8.5, title_fontsize=10,
           bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()


## Word Cloud Visualization of Job Titles by AI Exposure

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from wordcloud import STOPWORDS
from collections import Counter
import pandas as pd
import re

fig, axes = plt.subplots(1, 3, figsize=(24, 8))
categories = ['high ai expose', 'low ai expose', 'non-ai expose']
colormaps = ['Reds', 'Blues', 'Greys']

print("===== Top 5 Frequency Words in Titles =====")

for i, cat in enumerate(categories):
    titles = final_df[final_df['ai_exposure_label'] == cat]['title'].astype(str)
    combined_text = " ".join(titles).lower()
    all_stopwords = set(STOPWORDS)
    words = re.findall(r'\b\w+\b', combined_text)
    filtered_words = [w for w in words if w not in all_stopwords and len(w) > 1]
    word_counts = Counter(filtered_words)
    top_5 = word_counts.most_common(5)

    print(f"\n[{cat.upper()}]:")
    for word, freq in top_5:
        print(f" - {word}: {freq}")
    text_for_wc = " ".join(titles)
    if len(text_for_wc.strip()) > 5:
        wc = WordCloud(
            background_color='white',
            width=800,
            height=800,
            max_words=50,
            colormap=colormaps[i],
            stopwords=all_stopwords,
            collocations=True,
            min_font_size=10
        ).generate(text_for_wc)

        axes[i].imshow(wc, interpolation='bilinear')
        axes[i].set_title(f'Job Titles: {cat.upper()}', fontsize=20, fontweight='bold', pad=20)
    else:
        axes[i].text(0.5, 0.5, f'No Titles Found for\n{cat}', ha='center', fontsize=16)

    axes[i].axis('off')

plt.tight_layout()
plt.show()

## Word Cloud Visualization of Skills by AI Exposure

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import pandas as pd
from collections import Counter
import ast

def get_skill_counts(series):
    all_skills = []
    for s in series.dropna():
        if isinstance(s, str) and s.startswith('['):
            try:
                s = ast.literal_eval(s)
            except:
                continue
        if isinstance(s, list):
            processed_skills = [skill.replace(" ", "_") for skill in s if skill]
            all_skills.extend(processed_skills)
    return Counter(all_skills)

fig, axes = plt.subplots(1, 3, figsize=(24, 8))
categories = ['high ai expose', 'low ai expose', 'non-ai expose']
colormaps = ['Reds', 'Blues', 'Greens']

print("===== Top 5 Frequency Skills by Category =====")

for i, cat in enumerate(categories):
    cat_skills = final_df[final_df['ai_exposure_label'] == cat]['skill_name']
    word_counts = get_skill_counts(cat_skills)

    top_5 = word_counts.most_common(5)
    print(f"\n[{cat.upper()}]:")
    if top_5:
        for skill, freq in top_5:
            print(f" - {skill.replace('_', ' ')}: {freq}")
    else:
        print(" - No data available")

    if word_counts:
        wc = WordCloud(
            background_color='white',
            width=800,
            height=800,
            max_words=200,
            colormap=colormaps[i],
            min_font_size=2,
            relative_scaling=0.3,
            prefer_horizontal=0.9,
            regexp=r"\w[\w']+",
            collocations=False
        ).generate_from_frequencies(word_counts)

        axes[i].imshow(wc, interpolation='bilinear')
        axes[i].set_title(f'Skills: {cat.upper()}', fontsize=20, fontweight='bold', pad=20)
    else:
        axes[i].text(0.5, 0.5, 'No Skill Data', ha='center', fontsize=16)

    axes[i].axis('off')

plt.tight_layout()
plt.show()